In [13]:
# Standard setup for Kedro + Jupyter integration
import os
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession
from kedro.framework.project import configure_project

# 1. Find the project root (assuming notebook is in notebooks/ folder)
project_path = Path.cwd().parent  # Adjust if your notebook is deeper
print(project_path)

# 2. Bootstrap the Kedro project
# if not (project_path / ".kedro.yml").exists():
#     raise ValueError(f"Kedro project not found at {project_path}")

bootstrap_project(project_path)
configure_project(project_path.name)

# 3. Create a session and load the catalog
session = KedroSession.create(project_path=project_path)
context = session.load_context()

# 4. Make catalog available
catalog = context.catalog

/Users/vishalsharma/Downloads/urban_company/uc_no_response


[04/20/25 18:44:15] INFO     Registering new custom resolver: 'km.random_name'                    ]8;id=241752;file:///Users/vishalsharma/miniforge3/envs/uc_env/lib/python3.10/site-packages/kedro_mlflow/framework/hooks/mlflow_hook.py\mlflow_hook.py]8;;\:]8;id=11657;file:///Users/vishalsharma/miniforge3/envs/uc_env/lib/python3.10/site-packages/kedro_mlflow/framework/hooks/mlflow_hook.py#65\65]8;;\

                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=658131;file:///Users/vishalsharma/miniforge3/envs/uc_env/lib/python3.10/site-packages/kedro_telemetry/plugin.py\plugin.py]8;;\:]8;id=824706;file:///Users/vishalsharma/miniforge3/envs/uc_env/lib/python3.10/site-packages/kedro_telemetry/plugin.py#233\233]8;;\
                             the product. No personal data or IP addresses are stored on our side. If              
                             you want to opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK`              
                             environment variables, or create a `.telemetry` file in the current                   
                             working directory with the contents `consent: false`. Read more at                    
                             https://docs.kedro.org/en/stable/configuration/telemetry.html                         

In [14]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

In [15]:
# load the data
df = catalog.load("raw_data")

[04/20/25 18:44:16] INFO     Loading data from raw_data (CSVDataset)...                         ]8;id=279698;file:///Users/vishalsharma/miniforge3/envs/uc_env/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=118045;file:///Users/vishalsharma/miniforge3/envs/uc_env/lib/python3.10/site-packages/kedro/io/data_catalog.py#401\401]8;;\

In [16]:
# check for null 
df.isnull().sum() * 100 / len(df)


Unnamed: 0                          0.000
NR_experienced_on_request           0.000
no_of_supercats_tried_last_30d     15.075
no_of_supercats_tried_last_90d     15.075
no_of_supercats_tried_last_365d    15.075
no_of_deliveries_last_30d          15.075
no_of_deliveries_last_90d          15.075
no_of_deliveries_last_365d         15.075
service_deliverd_aov_last_30d      58.910
service_deliverd_aov_last_90d      37.230
service_deliverd_aov_last_365d     20.735
most_deliverd_supercat             15.075
income_segment                     75.295
gender                              0.085
user_state_platform                15.075
days_on_platform                   15.075
NR_experienced_last_365d            0.085
revenues_next_6_months              7.610
request_id                          0.000
dtype: float64

In [17]:
threshold = 0.8 * len(df.columns)

# Drop rows where more than 80% of features are NULL
df_cleaned = df.dropna(thresh=len(df.columns) - threshold + 1)

In [18]:
df_cleaned.isnull().sum() * 100 / len(df_cleaned)


Unnamed: 0                          0.000000
NR_experienced_on_request           0.000000
no_of_supercats_tried_last_30d     15.002752
no_of_supercats_tried_last_90d     15.002752
no_of_supercats_tried_last_365d    15.002752
no_of_deliveries_last_30d          15.002752
no_of_deliveries_last_90d          15.002752
no_of_deliveries_last_365d         15.002752
service_deliverd_aov_last_30d      58.875044
service_deliverd_aov_last_90d      37.176600
service_deliverd_aov_last_365d     20.667567
most_deliverd_supercat             15.002752
income_segment                     75.273983
gender                              0.000000
user_state_platform                15.002752
days_on_platform                   15.002752
NR_experienced_last_365d            0.000000
revenues_next_6_months              7.531402
request_id                          0.000000
dtype: float64

In [19]:
df_cleaned[['request_id', 
            'service_deliverd_aov_last_30d', 'no_of_deliveries_last_30d', 
            'service_deliverd_aov_last_90d', 'no_of_deliveries_last_90d', 
            'service_deliverd_aov_last_365d', 'no_of_deliveries_last_365d', 
            ]].sample(5)

,request_id,service_deliverd_aov_last_30d,no_of_deliveries_last_30d,service_deliverd_aov_last_90d,no_of_deliveries_last_90d,service_deliverd_aov_last_365d,no_of_deliveries_last_365d
4041,303ed10b-ee4d-4c98-9371-1f6d99c46130,1448.0,1.0,1448.0,1.0,1448.000000,1.0
9574,5d92cd27-6ae9-4337-80ef-03b31fb995e4,NaN,NaN,NaN,NaN,NaN,NaN
3667,ca2d04ac-fa36-4595-a944-6fea31fe5c8c,NaN,0.0,457.5,2.0,862.333333,12.0
7411,677dfaa4-db86-43df-9ad1-9bc08c341854,2015.0,1.0,2015.0,1.0,2015.000000,1.0
12126,ea566327-5a39-43e3-9c5b-b92d2d3dcf48,NaN,0.0,298.0,1.0,578.777778,9.0


In [20]:
df_cleaned.loc[df_cleaned['no_of_deliveries_last_30d'] == 0, 'service_deliverd_aov_last_30d'] = 0
df_cleaned.loc[df_cleaned['no_of_deliveries_last_90d'] == 0, 'service_deliverd_aov_last_90d'] = 0
df_cleaned.loc[df_cleaned['no_of_deliveries_last_365d'] == 0, 'service_deliverd_aov_last_365d'] = 0

In [21]:
df_cleaned.isnull().sum() * 100 / len(df_cleaned)


Unnamed: 0                          0.000000
NR_experienced_on_request           0.000000
no_of_supercats_tried_last_30d     15.002752
no_of_supercats_tried_last_90d     15.002752
no_of_supercats_tried_last_365d    15.002752
no_of_deliveries_last_30d          15.002752
no_of_deliveries_last_90d          15.002752
no_of_deliveries_last_365d         15.002752
service_deliverd_aov_last_30d      15.002752
service_deliverd_aov_last_90d      15.002752
service_deliverd_aov_last_365d     15.002752
most_deliverd_supercat             15.002752
income_segment                     75.273983
gender                              0.000000
user_state_platform                15.002752
days_on_platform                   15.002752
NR_experienced_last_365d            0.000000
revenues_next_6_months              7.531402
request_id                          0.000000
dtype: float64

In [22]:
df_cleaned[df_cleaned['no_of_supercats_tried_last_30d'].isnull()][['no_of_supercats_tried_last_30d', 'no_of_supercats_tried_last_90d', 'no_of_supercats_tried_last_365d']]

,no_of_supercats_tried_last_30d,no_of_supercats_tried_last_90d,no_of_supercats_tried_last_365d
2,NaN,NaN,NaN
7,NaN,NaN,NaN
14,NaN,NaN,NaN
17,NaN,NaN,NaN
19,NaN,NaN,NaN
...,...,...,...
19980,NaN,NaN,NaN
19983,NaN,NaN,NaN
19989,NaN,NaN,NaN
19990,NaN,NaN,NaN


In [23]:
coexisting_nulls = [
    'no_of_supercats_tried_last_30d', 'no_of_supercats_tried_last_90d', 'no_of_supercats_tried_last_365d',
    'no_of_deliveries_last_30d', 'no_of_deliveries_last_90d', 'no_of_deliveries_last_365d',
    'service_deliverd_aov_last_30d', 'service_deliverd_aov_last_90d', 'service_deliverd_aov_last_365d', 
]
df_cleaned.dropna(inplace=True, subset=coexisting_nulls)

In [24]:
df_cleaned.isnull().sum() * 100 / len(df_cleaned)


Unnamed: 0                          0.000000
NR_experienced_on_request           0.000000
no_of_supercats_tried_last_30d      0.000000
no_of_supercats_tried_last_90d      0.000000
no_of_supercats_tried_last_365d     0.000000
no_of_deliveries_last_30d           0.000000
no_of_deliveries_last_90d           0.000000
no_of_deliveries_last_365d          0.000000
service_deliverd_aov_last_30d       0.000000
service_deliverd_aov_last_90d       0.000000
service_deliverd_aov_last_365d      0.000000
most_deliverd_supercat              0.000000
income_segment                     73.217545
gender                              0.000000
user_state_platform                 0.000000
days_on_platform                    0.000000
NR_experienced_last_365d            0.000000
revenues_next_6_months              5.398881
request_id                          0.000000
dtype: float64

In [26]:
df_cleaned.shape[0]/df.shape[0]*100

84.925